**Размытие по Гауссу** заменяет каждый пиксель взвешенным средним его самого и соседних пикселей. Близкие пиксели получают большой вес, далёкие — маленький.

В результате резкие изменения яркости становятся плавными, а мелкий шум и небольшие детали ослабляются.

### Основная идея

Чёрно-белое изображение можно представить как матрицу чисел. Каждое число — яркость пикселя, например от $0$ до $255$:

- $0$ — чёрный;
- $255$ — белый;
- промежуточные числа — оттенки серого.

Чтобы пересчитать один пиксель, вокруг него берут небольшое окно, например $3\times3$. На окно накладывают **ядро** — матрицу весов такого же размера.

Простое приближение гауссова ядра:
$$
K=\frac1{16}
\begin{pmatrix}
1&2&1\\
2&4&2\\
1&2&1
\end{pmatrix}.
$$

Смысл коэффициентов:

- центральный пиксель имеет вес $4/16$;
- соседи сверху, снизу, слева и справа — по $2/16$;
- диагональные соседи — по $1/16$;
- сумма всех весов равна $1$.

Центр влияет сильнее всего, потому что он ближе всего к пересчитываемой точке.

### Что именно перемещается по изображению

Для всего изображения строится **одно ядро**. Его коэффициенты не пересчитываются при каждом сдвиге. Ядро целиком передвигается по изображению слева направо и сверху вниз.

Под центром понимается **центральная клетка ядра**, а не центр изображения. В ядре $3\times3$ это клетка со значением $4/16$:
$$
\frac1{16}
\begin{pmatrix}
1&2&1\\
2&\boxed{4}&2\\
1&2&1
\end{pmatrix}.
$$

При вычислении нового значения пикселя $(i,j)$ центр ядра совмещается именно с этим пикселем:
$$
\begin{pmatrix}
I(i-1,j-1)&I(i-1,j)&I(i-1,j+1)\\
I(i,j-1)&\boxed{I(i,j)}&I(i,j+1)\\
I(i+1,j-1)&I(i+1,j)&I(i+1,j+1)
\end{pmatrix}.
$$

Результат взвешенного сложения записывается в пиксель $I_{\text{blur}}(i,j)$. Затем ядро сдвигается на один пиксель вправо, его центр совмещается с $I(i,j+1)$, и вычисляется $I_{\text{blur}}(i,j+1)$.

Координаты $(x,y)$ в функции Гаусса — это не координаты пикселя во всём изображении. Это **локальные смещения клетки внутри ядра относительно его центра**. Для ядра $3\times3$ они всегда имеют вид
$$
\begin{pmatrix}
(-1,-1)&(0,-1)&(1,-1)\\
(-1,0)&\boxed{(0,0)}&(1,0)\\
(-1,1)&(0,1)&(1,1)
\end{pmatrix}.
$$

После перемещения ядра эти локальные координаты остаются теми же. Меняется только участок изображения, на который наложено ядро.

### Пересчёт одного пикселя

Пусть центральный пиксель имеет яркость $100$, а его соседи — $200$:
$$
\begin{pmatrix}
200&200&200\\
200&100&200\\
200&200&200
\end{pmatrix}.
$$

Умножаем каждое значение на соответствующий вес и складываем:
$$
\frac{
4\cdot1\cdot200+
4\cdot2\cdot200+
4\cdot100
}{16}=175.
$$

Новая яркость центрального пикселя равна $175$. Она приблизилась к яркости соседей, но сам центральный пиксель сохранил повышенный вес.

Алгоритм повторяет эту операцию для каждого пикселя изображения. Соседние окна пересекаются, поэтому яркость постепенно распространяется между близкими пикселями — изображение выглядит размытым.

### Свёртка

Операция наложения ядра, поэлементного умножения и сложения называется **свёрткой**.

Для изображения $I$ результат в точке $(i,j)$ равен
$$
I_{\text{blur}}(i,j)=
\sum_{u=-r}^{r}\sum_{v=-r}^{r}
K(u,v)I(i+u,j+v),
$$
где:

- $K$ — ядро размытия;
- $(i,j)$ — пересчитываемый пиксель;
- $(u,v)$ — смещение относительно него;
- $r$ — радиус ядра; для ядра $3\times3$ значение $r=1$.

Это та же операция, что и в числовом примере: два знака суммы только сокращённо обозначают перебор всех строк и столбцов окна.

### Откуда берутся веса

Точные веса рассчитывают по двумерной функции Гаусса:
$$
G(x,y)=\frac{1}{2\pi\sigma^2}
e^{-\frac{x^2+y^2}{2\sigma^2}},
$$
где:

- $(x,y)$ — смещение клетки ядра относительно центра;
- $\sigma$ — параметр, определяющий ширину распределения.

Для центра $x=0$, $y=0$, поэтому вес максимален. Чем дальше клетка от центра, тем больше $x^2+y^2$ и тем меньше её вес.

При построении дискретного ядра:

1. Выбирают нечётный размер, например $3\times3$, $5\times5$ или $7\times7$.
2. Для каждой клетки подставляют её смещение $(x,y)$ в функцию Гаусса.
3. Все полученные значения делят на их общую сумму.

Последний шаг делает сумму ядра равной $1$. Без нормировки однородное изображение после обработки становилось бы светлее или темнее.

### Обязательна ли нормировка

Для обычного гауссова размытия ядро нормируют обязательно:
$$
\sum_{u,v}K(u,v)=1.
$$

Пусть все пиксели окна имеют яркость $10$, а сумма ненормированных весов равна $S$. Тогда результат свёртки будет
$$
10\sum_{u,v}K(u,v)=10S.
$$

Следовательно:

- если $S=1$, однородная область сохранит яркость $10$;
- если $S>1$, изображение станет светлее;
- если $S<1$, изображение станет темнее.

Множитель $1/(2\pi\sigma^2)$ нормирует непрерывную функцию Гаусса на всей бесконечной плоскости. Дискретное ядро содержит только конечное число её значений, поэтому их сумма обычно не равна точно $1$. По этой причине после построения ядра его элементы всё равно делят на их фактическую сумму.

Нормировка не является обязательным правилом для любой свёртки. Например, ядра поиска границ специально имеют другую сумму. Но если задача состоит именно в усреднении и размытии без изменения общей яркости, сумма весов должна быть равна $1$.

### Роль $\sigma$ и размера ядра

$\sigma$ определяет, насколько быстро уменьшается влияние соседей:

- маленькое $\sigma$ — почти вся масса сосредоточена в центре, размытие слабое;
- большое $\sigma$ — заметный вес получают более далёкие пиксели, размытие сильнее.

Так как чем больше $\sigma$ тем меньше число мы получаем при возведении $e$ ``

Размер ядра определяет, насколько далеко алгоритм вообще смотрит от текущего пикселя. Большому $\sigma$ требуется большое ядро, иначе значительная часть распределения будет отброшена.  

Часто используют размер
$$
k=2\lceil3\sigma\rceil+1.
$$

Он охватывает приблизительно по $3\sigma$ в каждую сторону. Это практическое правило, а не обязательное условие.

### Почему размытие можно выполнить в два прохода

Двумерная функция Гаусса разделяется на произведение двух одномерных функций:
$$
G(x,y)=g(x)g(y).
$$

Поэтому вместо одного ядра $k\times k$ можно:

1. Размыть изображение по горизонтали ядром $1\times k$.
2. Размыть полученный результат по вертикали ядром $k\times1$.

Результат будет тем же, но для одного пикселя потребуется примерно $2k$ операций вместо $k^2$.

### Обработка границ изображения

Окно крайних пикселей частично выходит за изображение. Недостающие значения можно получить несколькими способами:

- продолжить крайнее значение;
- зеркально отразить изображение;
- считать отсутствующие пиксели нулевыми.

Нулевое дополнение может создавать тёмную рамку, поэтому чаще применяют отражение или повторение края.

### Размытие цветного изображения

В RGB-изображении каждый пиксель содержит не одну яркость, а три компоненты:
$$
I(i,j)=\bigl(R(i,j),G(i,j),B(i,j)\bigr).
$$

Одно и то же гауссово ядро независимо применяется к каждому каналу:
$$
R_{\text{blur}}(i,j)=\sum_{u,v}K(u,v)R(i+u,j+v),
$$
$$
G_{\text{blur}}(i,j)=\sum_{u,v}K(u,v)G(i+u,j+v),
$$
$$
B_{\text{blur}}(i,j)=\sum_{u,v}K(u,v)B(i+u,j+v).
$$

После этого три результата снова объединяются в один цветной пиксель. Каналы не смешиваются друг с другом: красные компоненты усредняются только с красными, зелёные — с зелёными, синие — с синими.

Например, если для упрощения два соседних пикселя имеют одинаковые веса $1/2$ и цвета
$$
(255,0,0) \quad\text{и}\quad (0,0,255),
$$
то результат равен
$$
\frac12(255,0,0)+\frac12(0,0,255)
=(127{,}5,0,127{,}5).
$$

Получается фиолетовый цвет, потому что независимо усреднились красный и синий каналы.

Для изображения с прозрачностью RGBA альфа-канал можно размывать тем же способом либо оставить без изменения — выбор зависит от задачи. При работе через OpenCV нужно помнить, что по умолчанию каналы хранятся в порядке BGR, но `GaussianBlur` всё равно обрабатывает каждый из них независимо.

### Применение и ограничения

Размытие по Гауссу применяют для:

- подавления мелкого случайного шума;
- удаления несущественных деталей;
- подготовки изображения к уменьшению размера;
- построения изображений разных масштабов;
- предварительной обработки перед [[Алгоритм Канни|поиском границ методом Канни]].

Недостатки:

- вместе с шумом размываются полезные контуры;
- плохо удаляет одиночные очень яркие или тёмные точки — для такого шума лучше подходит медианный фильтр;
- большое ядро требует больше вычислений, хотя разделимость фильтра уменьшает затраты.

В OpenCV размытие выполняется так:

In [1]:
import cv2

blurred = cv2.GaussianBlur(
    image,
    ksize=(5, 5),
    sigmaX=1.0,
)

Здесь `(5, 5)` — размер ядра, а `sigmaX` — значение $\sigma$ по горизонтали. Если отдельное `sigmaY` не указано, OpenCV использует связанное с `sigmaX` значение.

### Полный пример для изображения $5\times5$

Возьмём чёрно-белое изображение, в котором фон имеет яркость $10$, а один пиксель — яркость $100$:
$$
I=
\begin{pmatrix}
10&10&10&10&10\\
10&10&10&10&10\\
10&10&100&10&10\\
10&10&10&10&10\\
10&10&10&10&10
\end{pmatrix}.
$$

Используем ядро $3\times3$ и $\sigma=1$. Для начала получим его коэффициенты из функции Гаусса.

Общий множитель $1/(2\pi\sigma^2)$ можно пока не учитывать: он одинаков для всех клеток и сократится при нормировке. Поэтому вычисляем
$$
q(x,y)=e^{-\frac{x^2+y^2}{2}}.
$$

В ядре $3\times3$ существуют три типа клеток:

- центр $(0,0)$:
  $$
  q(0,0)=e^0=1;
  $$
- четыре соседа по стороне, например $(1,0)$:
  $$
  q(1,0)=e^{-1/2}\approx0{,}606531;
  $$
- четыре диагональных соседа, например $(1,1)$:
  $$
  q(1,1)=e^{-1}\approx0{,}367879.
  $$

Сумма всех сырых весов равна
$$
S=1+4\cdot0{,}606531+4\cdot0{,}367879
\approx4{,}897640.
$$

Делим каждый сырой вес на $S$:

- вес центра:
  $$
  c=\frac{1}{4{,}897640}\approx0{,}204180;
  $$
- вес соседа по стороне:
  $$
  s=\frac{0{,}606531}{4{,}897640}\approx0{,}123841;
  $$
- вес диагонального соседа:
  $$
  d=\frac{0{,}367879}{4{,}897640}\approx0{,}075114.
  $$

Получаем ядро
$$
K=
\begin{pmatrix}
d&s&d\\
s&c&s\\
d&s&d
\end{pmatrix}
=
\begin{pmatrix}
0{,}075114&0{,}123841&0{,}075114\\
0{,}123841&0{,}204180&0{,}123841\\
0{,}075114&0{,}123841&0{,}075114
\end{pmatrix}.
$$

Проверка нормировки:
$$
c+4s+4d
=0{,}204180+4\cdot0{,}123841+4\cdot0{,}075114
\approx1.
$$

Строки и столбцы здесь нумеруются начиная с $1$. В этом примере не будем дополнять изображение по краям. Поэтому центр ядра может находиться только в строках $2,3,4$ и столбцах $2,3,4$. Из изображения $5\times5$ получится результат $3\times3$.

Если бы все девять пикселей окна имели яркость $10$, результат был бы
$$
10\sum K=10\cdot1=10.
$$

Яркий пиксель имеет значение $100$ вместо $10$. Значит, он добавляет к результату ещё
$$
(100-10)\cdot k=90k,
$$
где $k$ — вес той клетки ядра, которая в текущем положении лежит над ярким пикселем.

#### Первая строка результата

Центр ядра находится над пикселем $I(2,2)$. Яркий пиксель $I(3,3)=100$ попадает в правый нижний угол ядра, где вес равен $d$:
$$
B(2,2)=10+90d
=10+90\cdot0{,}075114
\approx16{,}760.
$$

Сдвигаем ядро вправо: его центр находится над $I(2,3)$. Яркий пиксель попадает в нижнюю среднюю клетку с весом $s$:
$$
B(2,3)=10+90s
=10+90\cdot0{,}123841
\approx21{,}146.
$$

Ещё один сдвиг вправо: центр находится над $I(2,4)$. Яркий пиксель попадает в левый нижний угол с весом $d$:
$$
B(2,4)=10+90d\approx16{,}760.
$$

#### Вторая строка результата

Для центра над $I(3,2)$ яркий пиксель находится справа от центра ядра:
$$
B(3,2)=10+90s\approx21{,}146.
$$

Для центра над самим ярким пикселем $I(3,3)$ используется центральный вес $c$. Полная сумма имеет вид
$$
\begin{aligned}
B(3,3)={}&10d+10s+10d\\
&+10s+100c+10s\\
&+10d+10s+10d.
\end{aligned}
$$

Собираем одинаковые слагаемые:
$$
B(3,3)=10(4d+4s)+100c.
$$

Так как $4d+4s=1-c$, получаем
$$
B(3,3)=10(1-c)+100c
=10+90c
\approx28{,}376.
$$

Для центра над $I(3,4)$ яркий пиксель находится слева от центра ядра:
$$
B(3,4)=10+90s\approx21{,}146.
$$

#### Третья строка результата

После следующего сдвига вниз получаются симметричные значения:
$$
B(4,2)=10+90d\approx16{,}760,
$$
$$
B(4,3)=10+90s\approx21{,}146,
$$
$$
B(4,4)=10+90d\approx16{,}760.
$$

Итоговая матрица без обработки границ:
$$
B\approx
\begin{pmatrix}
16{,}760&21{,}146&16{,}760\\
21{,}146&28{,}376&21{,}146\\
16{,}760&21{,}146&16{,}760
\end{pmatrix}.
$$

Исходный скачок яркости от $10$ до $100$ превратился в плавное пятно:

- центральное значение уменьшилось со $100$ до $28{,}376$;
- соседние значения увеличились;
- ближайшие по стороне пиксели получили больше яркости, чем диагональные;
- лишняя яркость не исчезла, а распределилась между соседними выходными пикселями согласно весам ядра.